# Retail Revenue Intelligence — Analysis Walkthrough

This notebook mirrors the production pipeline in `scripts/run_analysis.py` for portfolio presentation and interviews.

**Run the pipeline first** from the project root:
```bash
python scripts/run_analysis.py
```

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import load_config
from src.cleaning import load_raw, clean_transactions
from src.rfm import compute_rfm, segment_summary
from src.cohorts import build_cohort_table
from src.metrics import kpis, category_performance, channel_performance
from src.viz import setup_style

cfg = load_config()
setup_style(cfg)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Project root:", ROOT)

## 1. Load & clean

In [ ]:
tx, customers, products = load_raw(cfg)
orders, quality = clean_transactions(tx, customers, products, cfg)

print("Data quality report")
for k, v in quality.items():
    print(f"  {k}: {v}")

orders.head()

## 2. Executive KPIs

In [ ]:
kpi = kpis(orders)
pd.Series(kpi)

## 3. RFM segmentation

Recency / Frequency / Monetary scored 1–5. Personas map scores to actionable groups (Champions, At Risk, etc.).

In [ ]:
rfm = compute_rfm(orders, cfg)
seg = segment_summary(rfm)
seg

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_df = seg.sort_values("revenue", ascending=True)
ax.barh(plot_df["segment"], plot_df["revenue"], color="#1B4F72")
ax.set_title("Revenue by RFM segment")
ax.set_xlabel("Revenue")
plt.tight_layout()
plt.show()

## 4. Cohort retention

In [ ]:
retention, sizes = build_cohort_table(orders)
mat = retention.iloc[:, :12].tail(12)

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(mat, annot=True, fmt=".0f", cmap="YlGnBu", ax=ax)
ax.set_title("Cohort retention % (recent cohorts)")
ax.set_xlabel("Months since first purchase")
plt.tight_layout()
plt.show()

sizes.tail(6)

## 5. Category performance (ABC + margin)

In [ ]:
cats = category_performance(orders)
display(cats)

print("\nChannel mix")
display(channel_performance(orders))

## 6. Interview talking points

1. **Champions vs headcount** — small share of customers, outsized revenue → retention > acquisition for this cohort.
2. **At Risk still holds value** — win-back should rank by monetary, not spray discounts.
3. **ABC ≠ margin** — Electronics can be A on revenue and still need attach strategies for profit.
4. **Cohorts diagnose product/onboarding** — flat or falling M1 retention is an early-life problem, not a brand problem.
5. **Reproducibility** — seed + modules + config so stakeholders can re-run after every data refresh.